# Laboratorio 04: Algoritmo de Bernstein-Vazirani

Dado un oráculo que calcula $f(x) = s \cdot x \pmod{2}$ (producto escalar binario), BV determina el **secreto $s$** con **1 sola consulta** (clásico necesita $n$ consultas).

**Objetivo:** implementar el oráculo para distintos secretos y verificar la recuperación exacta.

**Prerequisito:** Módulo 05 (algoritmos), Lab 01 (Grover).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np
print('Entorno listo')

## 1. El algoritmo

```
|0⟩^n ──H──[ Oráculo f(x)=s·x ]──H──[ Medir ] → s
|1⟩   ──H──[      (ancilla)   ]
```

El oráculo es una función lineal: CNOT del qubit ancilla controlado por el qubit $i$ si $s_i=1$.

In [ ]:
def bv_oracle(s: str) -> QuantumCircuit:
    'Oraculo para f(x) = s.x mod 2. s es una cadena binaria.'
    n = len(s)
    qc = QuantumCircuit(n + 1, name=f'Oraculo_s={s}')  # n qubits + 1 ancilla
    # CNOT controlado por qubit i si s[i]=1
    for i, bit in enumerate(reversed(s)):  # Qiskit: qubit 0 es el LSB
        if bit == '1':
            qc.cx(i, n)  # qubit n = ancilla
    return qc

# Ejemplo con s='101'
oracle = bv_oracle('101')
print('Oraculo para s=101:')
print(oracle.draw('text'))

In [ ]:
def bv_circuit(s: str) -> QuantumCircuit:
    'Circuito completo de Bernstein-Vazirani.'
    n = len(s)
    qc = QuantumCircuit(n + 1, n)
    # Estado inicial ancilla |1>
    qc.x(n)
    # Hadamard a todos
    qc.h(range(n + 1))
    qc.barrier()
    # Oraculo
    qc.compose(bv_oracle(s), inplace=True)
    qc.barrier()
    # Hadamard a los n qubits de entrada
    qc.h(range(n))
    # Medir
    qc.measure(range(n), range(n))
    return qc

qc_bv = bv_circuit('101')
qc_bv.draw('text')

## 2. Verificar recuperación del secreto

In [ ]:
from qiskit.quantum_info import Statevector

# Probar con distintos secretos
secrets = ['0', '1', '101', '1100', '10110111']

print(f'{'Secreto s':>12} | {'Recuperado':>12} | Exito')
print('-' * 38)
for s in secrets:
    n = len(s)
    # Circuito sin medicion para usar Statevector
    qc = QuantumCircuit(n + 1)
    qc.x(n); qc.h(range(n + 1))
    qc.compose(bv_oracle(s), inplace=True)
    qc.h(range(n))
    sv = Statevector(qc)
    # El estado resultante en los n qubits de entrada es |s>
    probs = sv.probabilities(range(n))  # marginal sobre qubits 0..n-1
    # El estado con maxima probabilidad codifica s
    idx_max = np.argmax(probs)
    s_rec = format(idx_max, f'0{n}b')
    ok = s_rec == s
    print(f'{s:>12} | {s_rec:>12} | {"✅" if ok else "❌"}')

## 3. Comparativa clásica vs cuántica

In [ ]:
n_vals = range(1, 12)
t_clasico = list(n_vals)        # n consultas clasicas
t_cuantico = [1] * len(n_vals)  # siempre 1

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(n_vals, t_clasico,  'r-o', lw=2, ms=7, label='Clasico: O(n) consultas')
plt.plot(n_vals, t_cuantico, 'b-s', lw=2, ms=7, label='BV cuantico: O(1) consulta')
plt.xlabel('Longitud del secreto n'); plt.ylabel('Consultas al oraculo')
plt.title('Bernstein-Vazirani: ventaja cuantica exponencial')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('bv_comparativa.png', dpi=100); plt.show()
print('BV demuestra separacion exponencial entre clasico y cuantico.')